This notebook uses the shared random training (70%), validation (15%), and test (15%) splits prepared by `preprocessing_refined.ipynb` with seed 42. Run preprocessing first. Apply task filters and feature transformations to all three sets. Compare models on validation data, then evaluate the fitted models on test data without refitting.


In [ ]:
from pyspark.ml.classification import (
    DecisionTreeClassifier,
    LogisticRegression,
    RandomForestClassifier,
)
from pyspark.ml.feature import StringIndexer
from pyspark.sql import functions as F
from xgboost.spark import SparkXGBClassifier

from classification_helper import (
    assemble,
    check_label_coverage,
    classification_report,
    evaluate_models,
    evaluate_predictions,
    perform,
    print_results,
    setup_spark,
)


In [ ]:
spark = setup_spark("Classification_nopitch")

selected_types = ["FF", "SI", "SL", "CH", "FC", "ST", "CU", "FS", "KC"]
train = spark.read.parquet("./data/random_reduced_traintest/train")
validation = spark.read.parquet("./data/random_reduced_traintest/validation")
test = spark.read.parquet("./data/random_reduced_traintest/test")

train = train.filter(F.col("pitch_type").isin(selected_types))
validation = validation.filter(F.col("pitch_type").isin(selected_types))
test = test.filter(F.col("pitch_type").isin(selected_types))

print("Training observations:", train.count())
print("Validation observations:", validation.count())
print("Test observations:", test.count())
train.groupBy("game_year").count().orderBy("game_year").show()
validation.groupBy("game_year").count().orderBy("game_year").show()
test.groupBy("game_year").count().orderBy("game_year").show()


In [ ]:
# randomSplit does not guarantee that every class appears in training.
check_label_coverage(train, validation, test, "pitch_type")

label_indexer = StringIndexer(
    inputCol="pitch_type",
    outputCol="label",
)
idx_model = label_indexer.fit(train)
train = idx_model.transform(train)
validation = idx_model.transform(validation)
test = idx_model.transform(test)


In [ ]:
numerical = [
    "on_3b", "on_2b", "on_1b", "inning", "sz_top", "sz_bot",
    "bat_score", "fld_score", "bat_score_diff", "home_win_exp",
    "bat_win_exp", "age_pit", "age_bat", "n_thruorder_pitcher",
    "n_priorpa_thisgame_player_at_bat", "pitcher_days_since_prev_game",
    "batter_days_since_prev_game"
]
one_hot_encoded = [column for column in train.columns if column.endswith("_idx")]
features = numerical + one_hot_encoded

_, validation = assemble(train, validation, features)
train, test = assemble(train, test, features)


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        featuresCol="features",
        maxIter=100,
    ),
    "Decision Tree": DecisionTreeClassifier(
        featuresCol="features",
        labelCol="label",
        maxDepth=10,
        seed=42,
    ),
    "Random Forest": RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=100,
        maxDepth=10,
        seed=42,
    )
}


In [ ]:
metrics = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]
results, fit_models = perform(train, validation, models, metrics, "label")
predictions = {
    name: result["predictions"]
    for name, result in results.items()
}
print_results(results)


In [ ]:
# Evaluate fitted models on test data without fitting again.
print("\nTest results")
test_results = evaluate_models(test, fit_models, metrics, "label")
print_results(test_results)


In [ ]:
class_reports = {}
for name, prediction in predictions.items():
    per_class, macro_f1 = classification_report(prediction, "label")
    class_reports[name] = {"per_class": per_class, "macro_f1": macro_f1}
    print(f"\n{name}")
    per_class.show(30, truncate=False)
    print(f"Macro F1: {macro_f1:.4f}")


In [ ]:
# Evaluation metrics.
metrics = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]

# Mark training and validation rows for XGBoost.
train_xgb = train.withColumn("is_validation", F.lit(False))
val_xgb = validation.withColumn("is_validation", F.lit(True))

# Combine training and validation data for early stopping.
xgb_data = train_xgb.unionByName(val_xgb)

# Configure XGBoost.
xgb = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    validation_indicator_col="is_validation",
    num_workers=2,
    force_repartition=True,
    repartition_random_shuffle=True,
    max_depth=5,
    n_estimators=600,
    learning_rate=0.05,
    eval_metric="mlogloss",
    early_stopping_rounds=20,
)

# Fit XGBoost using validation data for early stopping.
xgb_model = xgb.fit(xgb_data)

# Predict training and validation data separately.
results_train = xgb_model.transform(train_xgb)
results_val = xgb_model.transform(val_xgb)

# Evaluate training and validation predictions.
train_scores = evaluate_predictions(results_train, metrics, "label")
validation_scores = evaluate_predictions(results_val, metrics, "label")
for metric in metrics:
    print(f"{metric}:")
    print(f"  Training:   {train_scores[metric]:.4f}")
    print(f"  Validation: {validation_scores[metric]:.4f}")
    print()


In [ ]:
print("\nXGBoost test results")
xgb_test_results = evaluate_models(test, {"XGBoost": xgb_model}, metrics, "label")
test_pred = xgb_test_results["XGBoost"]["predictions"]
print_results(xgb_test_results)


In [ ]:
per_class, macro_f1 = classification_report(test_pred, "label")
per_class.show(30, truncate=False)
print(f"\nMacro F1: {macro_f1:.4f}")
